# FactIndic 05 – Factuality verification layer

Checks every summary (all systems and tracks) against the **original monolingual article** (primary – the facts are identical across inputs, so this measures the summariser and not how well the checker reads code-mixed text) and against the model's actual input (secondary). (1) multilingual NLI entailment per summary sentence (mDeBERTa-v3, SummaC-style); (2) QA-based consistency (questions from the summary answered from the article by XLM-R); (3) entity, numeral and date alignment (IndicNER + cross-script matching) giving precision against the source and precision/recall/F1 against the reference; plus an error type per unsupported sentence.

| Setting (right-hand panel) | Value |
|---|---|
| Accelerator | **GPU T4 x2** or **GPU P100** |
| Internet | **On** |
| Inputs to attach | output of notebook 04 |
| Time (full profile) | 20–40 min |

Run it with **Save Version → Save & Run All (Commit)**. When it finishes, its output (folder `factindic_out`) is
ready to attach to the next notebook. Each notebook's output contains every earlier stage, so you only ever attach
the previous notebook.

**This version needs no Hugging Face login.** If the IndicNER model cannot be downloaded, it uses an open NER model instead and says so in the output.

In [1]:
# ── Setup: make the factindic code importable ─────────────────────────────────────────
# Kaggle: upload factindic-code.zip as a private dataset and attach it
#         (right-hand panel -> Add Input -> Your Work / Datasets -> factindic-code -> "+").
# Colab:  put factindic-code.zip in My Drive (and run the Google Drive cell first), or upload it to /content.
# Or:     push the repository to GitHub and paste its URL here (needs Internet).
REPO_URL = ""   # e.g. "https://github.com/<your-username>/factindic"

import glob, os, shutil, subprocess, sys, zipfile

ON_COLAB = "google.colab" in sys.modules
SRC = "/content/factindic_src" if ON_COLAB else "/kaggle/working/factindic_src"

def _is_pkg(d):
    return all(os.path.exists(os.path.join(d, f)) for f in ("__init__.py", "pipeline.py", "config.py"))

def _candidates():
    inits = glob.glob("/kaggle/input/**/__init__.py", recursive=True)
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    if ON_COLAB:  # shallow search only: Google Drive can be huge
        for base in ("/content", "/content/drive/MyDrive"):
            for depth in ("", "*/", "*/*/"):
                inits += glob.glob(f"{base}/{depth}factindic/__init__.py")
            zips += [z for z in glob.glob(f"{base}/*.zip") + glob.glob(f"{base}/*/*.zip")
                     if "factindic" in os.path.basename(z).lower()]
    return sorted(set(inits)), sorted(set(zips))

def _unzip(z):
    try:
        with zipfile.ZipFile(z) as zf:
            hits = [n for n in zf.namelist() if n.endswith("factindic/__init__.py")]
            if hits:
                shutil.rmtree(SRC, ignore_errors=True)
                zf.extractall(SRC)
                return os.path.dirname(os.path.dirname(os.path.join(SRC, sorted(hits, key=len)[-1])))
    except zipfile.BadZipFile:
        pass
    return None

def _find_code():
    inits, zips = _candidates()
    # 0) Colab: always unpack the newest zip from Drive, so replacing the zip there updates the code
    if ON_COLAB:
        for z in sorted(zips, key=os.path.getmtime, reverse=True):
            code = _unzip(z)
            if code:
                print("using", z)
                return code
    # 1) the repository folder (…/factindic/factindic/__init__.py)
    for p in inits:
        d = os.path.dirname(p)
        if _is_pkg(d) and os.path.basename(d) == "factindic":
            return os.path.dirname(d)
    # 2) only the inner package was uploaded (files directly in the dataset): copy it under the right name
    for p in inits:
        d = os.path.dirname(p)
        if _is_pkg(d):
            shutil.copytree(d, os.path.join(SRC, "factindic"), dirs_exist_ok=True)
            for req in glob.glob(os.path.join(os.path.dirname(d), "**", "requirements.txt"), recursive=True)[:1]:
                shutil.copy(req, SRC)
            return SRC
    # 3) a zip that nobody unpacked
    for z in zips:
        code = _unzip(z)
        if code:
            return code
    # 4) running locally from the repository
    for p in ("factindic", "../factindic"):
        if _is_pkg(p):
            return os.path.abspath(os.path.dirname(p) or ".")
    return None

CODE = _find_code()
if CODE is None and REPO_URL:
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, SRC], check=True)
    CODE = SRC
if CODE is None:
    where = (sorted(glob.glob("/content/*")) + sorted(glob.glob("/content/drive/MyDrive/*"))[:30]) if ON_COLAB else \
            (sorted(glob.glob("/kaggle/input/*")) + sorted(glob.glob("/kaggle/input/*/*"))[:30])
    how = ("Put factindic-code.zip in My Drive and run the Google Drive cell first (or upload the zip to /content)."
           if ON_COLAB else
           "Attach the dataset: right-hand panel -> Add Input -> Your Work (or Datasets) -> factindic-code -> '+'. "
           "If you uploaded it a minute ago, wait until Kaggle finishes processing it.")
    raise FileNotFoundError("factindic code not found.\n" + how + "\nWhat this notebook can see:\n  "
                            + ("\n  ".join(where) if where else "(nothing)"))
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
req = os.path.join(CODE, "requirements.txt")
pkgs = ["-r", req] if os.path.exists(req) else ["indic-transliteration", "wordfreq", "datasketch", "bert-score",
                                                "python-docx", "feedparser", "trafilatura", "peft", "openpyxl",
                                                "bitsandbytes"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
import factindic
from factindic import Config
from factindic import pipeline as P
print("factindic", factindic.__version__, "loaded from", CODE)

factindic 1.0.0 loaded from /kaggle/input/datasets/lkr1196/factindic-code/factindic


In [2]:
# ── Configuration ────────────────────────────────────────────────────────────────────
# Use the SAME profile in every notebook of the chain.
#   "full"  – the real experiment
#   "quick" – ~25 articles per language, to check the whole chain on Kaggle first
#   "mock"  – CPU dry run with stand-in models (no GPU, no downloads); never report its numbers
PROFILE = "full"
cfg = Config.make(PROFILE)
print(cfg.to_json())

{
  "profile": "full",
  "seed": 13,
  "langs": [
    "hi",
    "bn"
  ],
  "variants": [
    "mono",
    "cm",
    "roman"
  ],
  "data_source": "xlsum",
  "n_test": 200,
  "n_dev": 30,
  "n_train_cm": 60,
  "min_words": 120,
  "max_words_raw": 1200,
  "lead_words": 300,
  "max_sents": 14,
  "min_summary_words": 8,
  "max_summary_words": 60,
  "near_dup_threshold": 0.8,
  "natural_split": [
    0.7,
    0.1,
    0.2
  ],
  "scrape": true,
  "n_natural_per_lang": 100,
  "backend": "auto",
  "summarizer_model": "Qwen/Qwen3-4B-Instruct-2507",
  "codemix_model": "Qwen/Qwen3-4B-Instruct-2507",
  "openai_base_url": "",
  "openai_model": "",
  "openai_concurrency": 4,
  "max_model_len": 6144,
  "max_batch_tokens": 24000,
  "max_batch_size": 24,
  "repetition_penalty": 1.0,
  "workers": -1,
  "quantize": "auto",
  "summary_max_words": 40,
  "tracks": [
    "A",
    "B",
    "BV"
  ],
  "max_new_tokens_A": 448,
  "max_new_tokens_B": 1536,
  "seq2seq_models": [
    "mt5",
    "indicbart"
  ],
 

In [3]:
# ── Name-finding model (NER): use IndicNER if this account can download it, otherwise an open model ──
# ai4bharat/IndicNER is "gated" on Hugging Face (needs a logged-in account that accepted its terms).
# Nothing needs to be done here: without access, the notebook switches to an open model that covers
# Hindi and Bengali with the same PER / ORG / LOC labels.
# Optional: to use IndicNER, accept its terms on huggingface.co/ai4bharat/IndicNER and add your token
# as a Kaggle secret named HF_TOKEN (Add-ons -> Secrets).
import os
FALLBACK_NER = "Davlan/xlm-roberta-base-wikiann-ner"

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("Hugging Face token found in Kaggle Secrets.")
except Exception:
    print("No HF_TOKEN secret (that is fine).")

def _can_download(name):
    from huggingface_hub import hf_hub_download
    try:
        hf_hub_download(name, "config.json")
        return True, ""
    except Exception as e:
        return False, f"{type(e).__name__}: {str(e)[:200]}"

if PROFILE != "mock":
    ok, why = _can_download(cfg.ner_model)
    if not ok:
        print(f"Cannot download {cfg.ner_model} ({why.splitlines()[0]})")
        ok2, why2 = _can_download(FALLBACK_NER)
        assert ok2, (f"Could not download {FALLBACK_NER} either: {why2}\n"
                     "Check that Internet is ON in the right-hand panel (Session options).")
        cfg.ner_model = FALLBACK_NER
    print("NER model used in this run:", cfg.ner_model)
    os.makedirs(cfg.out_root, exist_ok=True)
    open(os.path.join(cfg.out_root, "ner_model_used.txt"), "w").write(cfg.ner_model + "\n")


No HF_TOKEN secret (that is fine).


Cannot download ai4bharat/IndicNER (GatedRepoError: 401 Client Error. (Request ID: Root=1-6afb00b0-0075920c5445a3035811f8ad;9c158085-6b73-409b-be8d-ce90dd57afe4))


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

NER model used in this run: Davlan/xlm-roberta-base-wikiann-ner


In [4]:
ver = P.run_stage05(cfg)

Stages available: 01_data, 02_codemix, 03_seq2seq, 04_llm
Loading Davlan/xlm-roberta-base-wikiann-ner, MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7, deepset/xlm-roberta-large-squad2


tokenizer_config.json:   0%|          | 0.00/398 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

... (20 outputs left out)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] XLMRobertaForQuestionAnswering LOAD REPORT from: deepset/xlm-roberta-large-squad2
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NER on 6893 summaries and 760 references
NLI against the original article
  NLI: 260596 premise-hypothesis pairs
NLI against the model's actual input
  NLI: 192013 premise-hypothesis pairs
QA-based check against the original article
  QA: 15723 questions
Verified 7998 summaries in 32.9 min


,lang,system,track,variant,nli_score,any_halluc,ent_prec_src,num_prec_src,qa_f1
0,bn,indicbart,-,cm,0.692681,0.520000,0.813962,0.960526,0.293974
1,bn,indicbart,-,mono,0.697629,0.350000,0.989003,0.978261,0.454634
2,bn,indicbart,-,natural,0.720565,0.278689,1.000000,1.000000,0.460581
3,bn,indicbart,-,roman,0.632099,0.750000,0.633838,1.000000,0.117733
4,bn,mt5,-,cm,0.824742,0.400000,0.784888,0.800000,0.237612
5,bn,mt5,-,mono,0.817854,0.315000,0.879630,0.916667,0.309309
6,bn,mt5,-,natural,0.806011,0.426230,0.750000,1.000000,0.305712
7,bn,mt5,-,roman,0.659966,0.720000,0.605060,0.812500,0.095812
8,bn,mt5-lora-cm,-,cm,0.810621,0.435000,0.797863,0.761905,0.240273
9,bn,mt5-lora-cm,-,mono,0.819008,0.330000,0.889286,0.791667,0.329015


In [5]:
from IPython.display import Markdown, display
display(Markdown(open(f"{cfg.out_root}/05_verify/verify_report.md", encoding="utf-8").read()))

# Verification (means, 0-1)

| lang | system | track | variant | nli_score | any_halluc | ent_prec_src | num_prec_src | qa_f1 |
|---|---|---|---|---|---|---|---|---|
| bn | indicbart | - | cm | 0.693 | 0.520 | 0.814 | 0.961 | 0.294 |
| bn | indicbart | - | mono | 0.698 | 0.350 | 0.989 | 0.978 | 0.455 |
| bn | indicbart | - | natural | 0.721 | 0.279 | 1.000 | 1.000 | 0.461 |
| bn | indicbart | - | roman | 0.632 | 0.750 | 0.634 | 1.000 | 0.118 |
| bn | mt5 | - | cm | 0.825 | 0.400 | 0.785 | 0.800 | 0.238 |
| bn | mt5 | - | mono | 0.818 | 0.315 | 0.880 | 0.917 | 0.309 |
| bn | mt5 | - | natural | 0.806 | 0.426 | 0.750 | 1.000 | 0.306 |
| bn | mt5 | - | roman | 0.660 | 0.720 | 0.605 | 0.812 | 0.096 |
| bn | mt5-lora-cm | - | cm | 0.811 | 0.435 | 0.798 | 0.762 | 0.240 |
| bn | mt5-lora-cm | - | mono | 0.819 | 0.330 | 0.889 | 0.792 | 0.329 |
| bn | mt5-lora-cm | - | natural | 0.813 | 0.393 | 0.733 | 1.000 | 0.302 |
| bn | mt5-lora-cm | - | roman | 0.706 | 0.690 | 0.606 | 0.833 | 0.083 |
| bn | qwen3-4b-instruct-2507 | A | cm | 0.785 | 0.370 | 0.914 | 0.928 | 0.288 |
| bn | qwen3-4b-instruct-2507 | A | mono | 0.819 | 0.290 | 0.982 | 0.913 | 0.345 |
| bn | qwen3-4b-instruct-2507 | A | natural | 0.829 | 0.377 | 0.908 | 0.890 | 0.425 |
| bn | qwen3-4b-instruct-2507 | A | roman | 0.761 | 0.685 | 0.737 | 0.938 | 0.137 |
| bn | qwen3-4b-instruct-2507 | B | cm | 0.833 | 0.670 | 0.823 | 0.915 | 0.443 |
| bn | qwen3-4b-instruct-2507 | B | mono | 0.860 | 0.300 | 0.989 | 0.933 | 0.602 |
| bn | qwen3-4b-instruct-2507 | B | natural | 0.828 | 0.410 | 0.952 | 0.962 | 0.603 |
| bn | qwen3-4b-instruct-2507 | B | roman | 0.777 | 0.832 | 0.694 | 0.953 | 0.236 |
| bn | qwen3-4b-instruct-2507 | BV | cm | 0.846 | 0.475 | 0.864 | 0.972 | 0.447 |
| bn | qwen3-4b-instruct-2507 | BV | mono | 0.862 | 0.220 | 0.983 | 0.983 | 0.604 |
| bn | qwen3-4b-instruct-2507 | BV | natural | 0.804 | 0.361 | 0.967 | 0.993 | 0.606 |
| bn | qwen3-4b-instruct-2507 | BV | roman | 0.784 | 0.765 | 0.698 | 0.970 | 0.237 |
| hi | indicbart | - | cm | 0.675 | 0.365 | 0.883 | 0.800 | 0.189 |
| hi | indicbart | - | mono | 0.686 | 0.290 | 0.974 | 0.818 | 0.241 |
| hi | indicbart | - | natural | 0.732 | 0.222 | 0.986 | 0.875 | 0.356 |
| hi | indicbart | - | roman | 0.716 | 0.610 | 0.726 | 0.971 | 0.174 |
| hi | mt5 | - | cm | 0.778 | 0.430 | 0.839 | 0.737 | 0.282 |
| hi | mt5 | - | mono | 0.784 | 0.390 | 0.867 | 0.778 | 0.347 |
| hi | mt5 | - | natural | 0.842 | 0.181 | 0.946 | 1.000 | 0.414 |
| hi | mt5 | - | roman | 0.686 | 0.675 | 0.621 | 0.872 | 0.087 |
| hi | mt5-lora-cm | - | cm | 0.772 | 0.445 | 0.833 | 0.765 | 0.295 |
| hi | mt5-lora-cm | - | mono | 0.778 | 0.395 | 0.881 | 0.824 | 0.330 |
| hi | mt5-lora-cm | - | natural | 0.839 | 0.181 | 0.943 | 1.000 | 0.413 |
| hi | mt5-lora-cm | - | roman | 0.680 | 0.770 | 0.539 | 0.833 | 0.074 |
| hi | qwen3-4b-instruct-2507 | A | cm | 0.802 | 0.355 | 0.943 | 0.863 | 0.350 |
| hi | qwen3-4b-instruct-2507 | A | mono | 0.793 | 0.290 | 0.993 | 0.910 | 0.375 |
| hi | qwen3-4b-instruct-2507 | A | natural | 0.827 | 0.125 | 1.000 | 0.984 | 0.529 |
| hi | qwen3-4b-instruct-2507 | A | roman | 0.825 | 0.465 | 0.847 | 0.966 | 0.282 |
| hi | qwen3-4b-instruct-2507 | B | cm | 0.821 | 0.640 | 0.846 | 0.929 | 0.491 |
| hi | qwen3-4b-instruct-2507 | B | mono | 0.836 | 0.290 | 0.996 | 0.931 | 0.598 |
| hi | qwen3-4b-instruct-2507 | B | natural | 0.856 | 0.250 | 0.981 | 0.959 | 0.611 |
| hi | qwen3-4b-instruct-2507 | B | roman | 0.827 | 0.755 | 0.768 | 0.960 | 0.363 |
| hi | qwen3-4b-instruct-2507 | BV | cm | 0.837 | 0.510 | 0.896 | 0.973 | 0.505 |
| hi | qwen3-4b-instruct-2507 | BV | mono | 0.855 | 0.200 | 0.996 | 0.986 | 0.606 |
| hi | qwen3-4b-instruct-2507 | BV | natural | 0.853 | 0.194 | 0.995 | 0.979 | 0.629 |
| hi | qwen3-4b-instruct-2507 | BV | roman | 0.815 | 0.615 | 0.810 | 0.983 | 0.415 |


In [6]:
import os
for root, dirs, files in os.walk(cfg.out_root):
    dirs[:] = [d for d in dirs if not d.startswith("_")]
    depth = root[len(cfg.out_root):].count(os.sep)
    if depth <= 1:
        print("  " * depth + os.path.basename(root) + "/", ", ".join(sorted(files))[:300])

factindic_out/ ner_model_used.txt
  05_verify/ config.json, verification.jsonl, verify_report.md
  01_data/ config.json, data_report.md, data_stats.csv, docs.jsonl
  03_seq2seq/ config.json, lora_training.json, summaries_seq2seq.jsonl
  04_llm/ config.json, env.json, llm_report.md, qg.jsonl, summaries_llm.jsonl
  02_codemix/ codemix_quality_check.xlsx, codemix_report.md, config.json, env.json, variants.jsonl
